# **Fine Tune With Adapters**

#### **Import Libraries**

In [40]:
import torch
import torch.nn as nn 
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoTokenizer, BertForSequenceClassification, BertTokenizer
from tqdm import tqdm
from torch.utils.data import Dataset
import math

#### **Dataset Configuration**

In [41]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [42]:
train_dataset = dataset['train']
test_dataset = dataset['test']

In [43]:
train_dataset[0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [44]:
label_list = []
for id , (text, label) in enumerate(train_dataset):
    label_list.append(train_dataset[id]['label'])

num_labels = len(set(label_list))
num_labels    

2

In [50]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [46]:
lmdb_labels = {0: "negatice review", 1: "positive review"}

In [47]:
label_col = train_dataset.select_columns('label')
label_col

Dataset({
    features: ['label'],
    num_rows: 25000
})

In [48]:
len(train_dataset)

25000

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cuda'

### **Define Dataset Class**

In [53]:
class ClassificationDataset(Dataset):
    
    def __init__(self,dataset,tokenizer: AutoTokenizer, max_length:int = 500):
        super().__init__()
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.dataset = dataset
    
    def __len__(self):
        return len(dataset)
    
    def __getitem__(self, index):
        item = self.dataset[index]
        
        encodings = self.tokenizer(
            item['text'],
            return_tensors = 'pt',
            max_length = self.max_length,
            truncation = True,
            padding = 'max_length'
        )
        label_tenosr = torch.tensor(item['label'], dtype=torch.int64)
        return encodings['input_ids'].squeeze(0).to(device), encodings['attention_mask'].squeeze(0).to(device), label_tenosr
        